# Multi-Session LeJEPA Experiment Runner

Train and evaluate the LeJEPA teacher model on multi-session neural spike data.

**Order of operations:**
1. Set `CONFIG_PATH` in the Config cell.
2. Run all cells top to bottom.
3. Results (CSV + plots) are saved under `results_out_path` from your config.
4. Training curves, test metrics, and latent-space UMAP also render inline below.

**Prerequisite:** Run the dataset pipeline first (`experiments/data/create_dataset.py`) so a Parquet exists at your config's `data_path`.

In [ ]:
import sys
from pathlib import Path

# Add repo root to sys.path so jepsyn is importable.
# This notebook lives in experiments/multi_session/, so root is two levels up.
REPO_ROOT = Path("../..").resolve()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Image, display

from multi_session import evaluate_model, load_and_prepare_data, save_results, train_lejepa
from jepsyn.utils import verify_config

print("Imports OK")

## Configuration

Point `CONFIG_PATH` at your experiment YAML. See `configs/lejepa_lif_visual_cortex.yaml` for a full template with all encoder, predictor, and training fields.

At minimum, `results_out_path` must be set in the config for checkpoints and plots to be saved to disk.

In [ ]:
CONFIG_PATH = Path("configs/lejepa_lif_visual_cortex.yaml")

config = verify_config(CONFIG_PATH)

print(f"Config loaded:      {CONFIG_PATH}")
print(f"data_path:          {config['data_path']}")
print(f"results_out_path:   {config.get('results_out_path', '(not set — results will not be saved to disk)')}")
print()
tcfg = config["training_config"]
mcfg = config["model_config"]
print(f"epochs:             {tcfg.get('epochs', 100)}")
print(f"batch_size:         {tcfg.get('batch_size', 32)}")
print(f"lr:                 {tcfg.get('lr', 1e-4)}")
print(f"mask_ratio:         {tcfg.get('mask_ratio', 0.5)}")
print(f"ema_momentum:       {tcfg.get('ema_momentum', 0.996)}")
print(f"lambd (SIGReg):     {tcfg.get('lambd', 0.05)}")
print()
print(f"d_model:            {mcfg.get('d_model', 256)}")
print(f"n_latents:          {mcfg.get('n_latents', 64)}")
print(f"max_time_ms:        {mcfg.get('max_time_ms', 400)}")

## Data Loading

Loads the Parquet from `data_path`, validates schema and integrity, then splits windows by `session_id` into train / val / test sets.

In [ ]:
train_loader, val_loader, test_loader, unit_maps = load_and_prepare_data(config)

print(f"\nSessions in unit_maps: {len(unit_maps)}")
print(f"Units per session:     min={min(len(m) for m in unit_maps.values())}, "
      f"max={max(len(m) for m in unit_maps.values())}")
print(f"Train batches:         {len(train_loader)}")
print(f"Val batches:           {len(val_loader)}")
print(f"Test batches:          {len(test_loader)}")

## LeJEPA Training

Trains the **context encoder** (online, gradients flow), **target encoder** (EMA copy, no gradients), and **predictor** (narrow Transformer).

- Loss: `(1 - λ) * MSE(h_pred, h_tgt) + λ * SIGReg(h_ctx, h_tgt)`
- After training, the checkpoint is saved to `<results_out_path>/lejepa_checkpoint.pt`.

In [ ]:
jepa_model, jepa_train_metrics = train_lejepa(config, train_loader, val_loader, unit_maps)
print("\nTraining complete.")
jepa_train_metrics.tail()

## Training Results

Saves `metrics.csv` and `training_curves.png` to `<results_out_path>/LeJEPA/training/`, then renders the curves inline.

In [ ]:
save_results(stage="LeJEPA", phase="training", metrics=jepa_train_metrics, config=config)

In [ ]:
# Inline training curves
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle("LeJEPA - Training Curves")

axes[0].plot(jepa_train_metrics["epoch"], jepa_train_metrics["train_loss"], label="train")
axes[0].plot(jepa_train_metrics["epoch"], jepa_train_metrics["val_loss"], label="val")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].set_title("Total Loss")
axes[0].legend()

axes[1].plot(jepa_train_metrics["epoch"], jepa_train_metrics["train_pred_loss"], label="pred loss")
axes[1].plot(jepa_train_metrics["epoch"], jepa_train_metrics["train_reg_loss"], label="reg loss")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Loss")
axes[1].set_title("Pred vs Reg Loss")
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# Full per-epoch metrics table
jepa_train_metrics

## LeJEPA Test Evaluation

Runs the trained model on the held-out test set (no masking — context encoder sees all events).

Metrics per batch:
- **`pred_loss`**: MSE between predicted and target mean-pooled representations `[B, D]`
- **`cos_similarity`**: cosine similarity between context and target representations

In [ ]:
jepa_test_metrics = evaluate_model(jepa_model, test_loader, stage="LeJEPA")

In [ ]:
save_results(stage="LeJEPA", phase="test", metrics=jepa_test_metrics, config=config)

# Display saved plots if results_out_path is set
results_path = config.get("results_out_path")
if results_path:
    test_bar_path = Path(results_path) / "LeJEPA" / "test" / "test_metrics.png"
    latent_path   = Path(results_path) / "LeJEPA" / "test" / "latent_space.png"
    if test_bar_path.exists():
        print("Test metrics bar chart:")
        display(Image(filename=str(test_bar_path)))
    if latent_path.exists():
        print("Latent space (UMAP):")
        display(Image(filename=str(latent_path)))

In [ ]:
# Inline: bar chart of mean test metrics
mean_metrics = jepa_test_metrics[["pred_loss", "cos_similarity"]].mean()

fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar(mean_metrics.index, mean_metrics.values, color=["steelblue", "seagreen"])
ax.set_ylabel("Value")
ax.set_title("LeJEPA - Mean Test Metrics")
for bar, val in zip(bars, mean_metrics.values):
    ax.text(bar.get_x() + bar.get_width() / 2, val + 0.002, f"{val:.4f}", ha="center", fontsize=10)
plt.tight_layout()
plt.show()

print("\nMean test metrics:")
print(mean_metrics.to_string())

In [ ]:
# Inline: UMAP of context latent vectors colored by session
try:
    import umap

    latent_vectors = np.vstack(jepa_test_metrics["h_ctx"].values)          # [N, D]
    session_labels = np.concatenate(jepa_test_metrics["session_ids"].values) # [N]

    print(f"Fitting UMAP on {latent_vectors.shape[0]} samples, dim={latent_vectors.shape[1]}...")
    reducer = umap.UMAP(n_components=2, random_state=42)
    embeddings2d = reducer.fit_transform(latent_vectors)

    fig, ax = plt.subplots(figsize=(8, 6))
    scatter = ax.scatter(
        embeddings2d[:, 0], embeddings2d[:, 1],
        c=session_labels, cmap="tab10", alpha=0.5, s=10,
    )
    plt.colorbar(scatter, ax=ax, label="Session ID")
    ax.set_title("LeJEPA - Latent Space (UMAP)")
    ax.set_xlabel("DIM 1")
    ax.set_ylabel("DIM 2")
    plt.tight_layout()
    plt.show()

except ImportError:
    print("umap-learn not installed. To generate UMAP plots, run:")
    print("  pip install umap-learn")